# How to Perform Curvature Analysis

This notebook computes FlowMap curvature on the Larry embedding, then shows total curvature, decomposed curvature, and a ternary-style acceleration decomposition.


## Setup


In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from flowmap.geometry import compute_flow_curvature
from flowmap.plot import plot_velocity_stream
from flowmap.utils import load_dataset

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tutorial").exists() and (PROJECT_ROOT.parent / "tutorial").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

The Larry data object is not bundled with the repository. Download it from Figshare and place `larry_flowmap_data.joblib` in the `tutorial/` folder before running this notebook.

Data link: [Larry Data Processed with FlowMap](https://figshare.com/articles/dataset/Larry_Data_Processed_with_FlowMap/32258700?file=64501479)


In [2]:
data = load_dataset(PROJECT_ROOT / "tutorial" / "larry_flowmap_data.joblib")
emb = data.embedder

print(f"X_emb: {emb.X_emb.shape}, V_emb: {emb.V_emb.shape}")
print(f"genes: {len(data.var_names)}")


X_emb: (49302, 2), V_emb: (49302, 2)
genes: 2000


## Compute Curvature

`compute_flow_curvature` evaluates the fitted manifold spline and velocity spline, then returns velocity, acceleration components, and curvature estimates.


In [ ]:
curv = compute_flow_curvature(emb)

k_total = curv["curvature"]["total"]
k_steer = curv["curvature"]["steer"]
k_surface = curv["curvature"]["surface"]

print(k_total.shape)


## Total Curvature

The total curvature is computed from the steering and surface components:

$$
\kappa_{\mathrm{total}} = \sqrt{\kappa_{\mathrm{steer}}^2 + \kappa_{\mathrm{surface}}^2}
$$


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_velocity_stream(
    emb.X_emb,
    spline=emb.spline_vf,
    scatter_color=np.clip(k_total, None, np.percentile(k_total, 99)),
    cmap="coolwarm",
    scatter_size=10,
    scatter_alpha=0.6,
    show_colorbar=True,
    ax=ax,
)
plt.show()


## Decomposed Curvature

FlowMap separates acceleration into flow, steering, and surface components. The curvature components are acceleration magnitudes normalized by squared speed:

$$
\kappa_{\mathrm{steer}} = \frac{\|A_{\mathrm{steer}}\|}{\|v\|^2}, \qquad
\kappa_{\mathrm{surface}} = \frac{\|A_{\mathrm{surface}}\|}{\|v\|^2}
$$


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, values, title in zip(
    axes,
    [k_total, k_steer, k_surface],
    ["total", "steer", "surface"],
):
    plot_velocity_stream(
        emb.X_emb,
        spline=emb.spline_vf,
        scatter_color=np.clip(values, None, np.percentile(values, 99)),
        cmap="coolwarm",
        scatter_size=8,
        scatter_alpha=0.6,
        ax=ax,
        title=title,
    )

plt.tight_layout()
plt.show()


## Ternary Decomposition

Each point summarizes the relative contribution of flow, steer, and surface acceleration. This plot uses the acceleration magnitudes before curvature normalization.


In [ ]:
A_flow = np.linalg.norm(curv["acceleration"]["flow"], axis=1)
A_steer = np.linalg.norm(curv["acceleration"]["steer"], axis=1)
A_surface = np.linalg.norm(curv["acceleration"]["surface"], axis=1)

total = A_flow + A_steer + A_surface + 1e-12
flow_frac = A_flow / total
steer_frac = A_steer / total
surface_frac = A_surface / total

# Barycentric coordinates for triangle vertices:
x = steer_frac + 0.5 * surface_frac
y = (np.sqrt(3) / 2) * surface_frac

fig, ax = plt.subplots(figsize=(5.8, 5.2))
ax.scatter(
    x, y,
    s=6,
    c=np.clip(k_total, None, np.percentile(k_total, 99)),
    cmap="coolwarm",
    alpha=0.35,
)

tri_x = [0, 1, 0.5, 0]
tri_y = [0, 0, np.sqrt(3) / 2, 0]
ax.plot(tri_x, tri_y, color="black", lw=1)
ax.text(-0.03, -0.04, "flow", ha="right", va="top")
ax.text(1.03, -0.04, "steer", ha="left", va="top")
ax.text(0.5, np.sqrt(3) / 2 + 0.04, "surface", ha="center", va="bottom")
ax.set_aspect("equal")
ax.set_axis_off()
plt.show()
